## >>> VERSION: 2026-07-28  ·  atgan-v2 (sid-fixed, all 500)  <<<
**atgan on the full 500-image benchmark (INSTRUMENTED).**

*v2 fix: the eval now passes `--sid ""` so it processes ALL 500 folders (the scripts
default `--sid=00199`, which would otherwise restrict the run to just folder 00199).*

Feed-forward restoration model (single forward pass) -> fast, **no batching**. Runs the eval via the
`get_ipython().system` shell magic on a **single GPU** (`CUDA_VISIBLE_DEVICES=0`, so no multi-GPU
duplication and no pydevd double-launch). Scores at a fixed **256x256** (same scorer as DiT/RCDNet,
so Stage 3 comparisons stay valid).

**Attach as inputs:** RaindropClarity code, the 500 dataset (`Drop/`+`Clear/`), the checkpoint bundle
(contains `atgan/epoch100.pth.tar`), and `score_pairs.py`+`model_stats.py`. **GPU T4 x2, Internet on.**

**After it finishes:** download `metrics.zip` (CSVs + stats) and `atgan_outputs.zip` (restored images).

In [ ]:
# --- setup: copy RaindropClarity (code + utils) and the scoring scripts ---
import os, shutil, glob
src = None
for d, _, files in os.walk('/kaggle/input'):
    if 'eval_diffusion_day_dit.py' in files:
        src = d; break
assert src, 'Could not find the RaindropClarity code under /kaggle/input'
shutil.copytree(src, '/kaggle/working/RaindropClarity', dirs_exist_ok=True)
os.chdir('/kaggle/working/RaindropClarity')
print('cwd =', os.getcwd())
for s in glob.glob('/kaggle/input/**/score_pairs.py', recursive=True):
    shutil.copy(s, 'score_pairs.py'); print('got', s)
for s in glob.glob('/kaggle/input/**/model_stats.py', recursive=True):
    shutil.copy(s, 'model_stats.py'); print('got', s)
os.makedirs('/kaggle/working/metrics', exist_ok=True)

In [ ]:
!pip install einops lpips timm -q

In [ ]:
# --- locate the 500 dataset (Drop/+Clear/) and this model's checkpoint ---
import glob, os, torch
DATA = None
for d, subs, _ in os.walk('/kaggle/input'):
    if 'Drop' in subs and 'Clear' in subs:
        DATA = d; break
assert DATA, "Couldn't find Drop/ and Clear/ under /kaggle/input"
ck = glob.glob('/kaggle/input/**/atgan/epoch100.pth.tar', recursive=True)
assert ck, "Couldn't find atgan/epoch100.pth.tar - attach the checkpoint bundle"
os.environ['DATA'] = DATA
os.environ['CKPT'] = ck[0]
n = len(glob.glob(os.path.join(DATA, 'Drop', '**', '*.png'), recursive=True))
print('DATA  :', DATA)
print('CKPT  :', ck[0])
print('images:', n, '(expect 500)')
print('GPU   :', torch.cuda.is_available(), ' device_count:', torch.cuda.device_count())

## Plumbing smoke test (20 images) — confirms score_pairs + LPIPS + CSV work

In [ ]:
!python score_pairs.py --pred "$DATA/Drop" --gt "$DATA/Clear" \
    --name _smoketest --out_dir /kaggle/working/metrics --limit 20
import pandas as pd
df = pd.read_csv('/kaggle/working/metrics/_smoketest_per_image.csv')
print('\nsmoke test rows:', len(df), '(expect 20)')
assert len(df) == 20, 'smoke test did not produce 20 rows - fix before continuing'
print('PLUMBING OK')

## Static resource stats — parameter count + model size

In [ ]:
!python model_stats.py --ckpt "$CKPT" --name atgan \
    --out_csv /kaggle/working/metrics/resource_static.csv
print(open('/kaggle/working/metrics/resource_static.csv').read())

## Point the model's config at the 500 dataset

In [ ]:
import re, pathlib, os
cfg = pathlib.Path('configs/daytime_256.yml')
t = cfg.read_text()
t = re.sub(r'data_dir:.*', 'data_dir: "%s"' % os.environ['DATA'], t)
t = re.sub(r'num_workers:.*', 'num_workers: 2', t)
cfg.write_text(t)
print('patched configs/daytime_256.yml  data_dir ->', os.environ['DATA'])

## Run atgan (fast, single-GPU) — timed, with peak-GPU logging

In [ ]:
# eval via the shell magic on ONE GPU (no duplication). Python timing + nvidia-smi poller wrap it.
import subprocess, time, os, glob
mem_log = '/kaggle/working/metrics/gpu_mem_atgan.csv'
lf = open(mem_log, 'w')
poller = subprocess.Popen(
    ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits', '-l', '1'],
    stdout=lf)
t0 = time.time()
rc = get_ipython().system(
    'CUDA_VISIBLE_DEVICES=0 python eval_diffusion_day_atgan.py --config daytime_256.yml --test_set atgan --resume "$CKPT" --sid ""')
elapsed = time.time() - t0
poller.terminate(); lf.close()
outs = sorted(glob.glob('results/RainDrop/atgan/*/output'), key=os.path.getmtime, reverse=True)
assert outs, 'no outputs produced - check the eval log above'
base = outs[0][:-len('output')]
os.environ['OUTD'] = base + 'output'
os.environ['GTD']  = base + 'gt'
os.environ['MEMLOG'] = mem_log
nimg = len(glob.glob(os.environ['OUTD'] + '/**/*.png', recursive=True))
os.environ['ELAPSED'] = str(elapsed); os.environ['NIMG'] = str(max(nimg, 1))
print('atgan done. elapsed:', round(elapsed, 1), 's | images:', nimg)

In [ ]:
# runtime resource stats -> resource_runtime.csv  (real full-500 total)
import csv as _csv, os
elapsed = float(os.environ['ELAPSED']); n = int(os.environ['NIMG'])
mv = [int(l.strip()) for l in open(os.environ['MEMLOG']) if l.strip().isdigit()]
peak = max(mv) if mv else -1
per = elapsed / n
row = {'model': 'atgan', 'per_image_s': round(per, 4),
       'throughput_img_per_s': round(1.0 / per, 4),
       'total_time_s': round(elapsed, 2), 'peak_gpu_mem_MB': peak, 'n_images': n}
rt = '/kaggle/working/metrics/resource_runtime.csv'
with open(rt, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=list(row.keys())); w.writeheader(); w.writerow(row)
print(row); print(open(rt).read())

In [ ]:
# score outputs at 256x256 -> atgan_per_image.csv + summary
import os
os.system('python score_pairs.py --pred "%s" --gt "%s" --name atgan --out_dir /kaggle/working/metrics'
          % (os.environ['OUTD'], os.environ['GTD']))
print(open('/kaggle/working/metrics/atgan_summary.txt').read())

In [ ]:
# collect restored images -> atgan_outputs/ and zip
import shutil, os, glob
shutil.rmtree('/kaggle/working/atgan_outputs', ignore_errors=True)
shutil.copytree(os.environ['OUTD'], '/kaggle/working/atgan_outputs')
os.system('cd /kaggle/working && rm -f atgan_outputs.zip && zip -r atgan_outputs.zip atgan_outputs -q')
n = len(glob.glob('/kaggle/working/atgan_outputs/**/*.png', recursive=True))
print('wrote atgan_outputs.zip (%d images)' % n)

## Package + verify

In [ ]:
import os
os.system('cd /kaggle/working && rm -f metrics.zip && zip -r metrics.zip metrics -q')
print('wrote metrics.zip')
print('DOWNLOAD from Output panel: metrics.zip (CSVs + stats), atgan_outputs.zip (restored images)')

In [ ]:
import os
try:
    import pandas as pd
except Exception:
    pd = None
m = '/kaggle/working/metrics'
def chk(p, expect=None):
    ok = os.path.exists(p); ex = ''
    if ok and p.endswith('.csv') and pd is not None:
        try:
            n = len(pd.read_csv(p)); ex = ' (%d rows)' % n
            if expect and n != expect: ex += '  << expected %d' % expect
        except Exception as e:
            ex = ' (%s)' % e
    print(('OK   ' if ok else 'MISS ') + p + ex)
chk(m + '/atgan_per_image.csv', 500)
chk(m + '/atgan_summary.txt')
chk(m + '/resource_static.csv')
chk(m + '/resource_runtime.csv')
print('\natgan_per_image.csv should have 500 rows. All metrics at 256x256 (matches DiT/RCDNet).')